In [16]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import random
import torch
import torch.nn as nn
import torch.optim.adam

In [29]:
env = gym.make("LunarLander-v3")

In [3]:
class MemoryReplay:
    def __init__(self, N):
        self.N = N
        self.buffer = []

    def push(self, state, action, reward, next_state, done):
        
        if(len(self.buffer) >= self.N):  self.buffer.pop(0)
        
        self.buffer.append([state, action, reward, next_state, done])

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

In [4]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, state):
        return self.net(state)

In [18]:
class EpsilonGreadyPolicy:
    def __init__(self, QNetwork, epsilon = 0.1):
        self.Q = QNetwork
        self.ep = epsilon

    def take_action(self, state):
        state = torch.tensor(state)
        if (random.random() < self.ep):
            action_prob = list(self.Q(state).detach().numpy())
            sample = random.sample(list(range(len(action_prob))), 1)[0]
            return np.int64(sample)
        else:
            return np.int64(self.Q(state).argmax())

    def set_epsilon(self, epsilon):
        self.ep = epsilon
        

In [ ]:
def train(env, replay_memory, Q, policy, criterion, optimizer, episodes = 200, batch_size = 30, learning_rate = 0.01, discount_factor = 0.9 ):

    ep_losses = []
    
    for i in range(episodes):
        observation, info = env.reset()
        total_reward = 0.0
        episode_done = False

        ep_loss = 0.0
        timestep = 0
        while (not episode_done):
            
            action = policy.take_action(observation)
            new_observation, reward, terminated, truncated, info  =  env.step(action)

            episode_done = terminated or truncated
            
            replay_memory.push(observation, action, reward, new_observation, episode_done)


            observation = new_observation

            if(len(replay_memory) >= batch_size):
                batch = replay_memory.sample(batch_size)
                st, at, rt, stnew, done = zip(*batch)
                st = torch.tensor(np.array(st), dtype=torch.float32)
                at = torch.tensor(at, dtype=torch.long)
                rt = torch.tensor(rt, dtype=torch.float32)
                stnew = torch.tensor(np.array(stnew), dtype=torch.float32)
                done = torch.tensor(done, dtype=torch.bool)
                
                qst = Q(st).gather(1, at.unsqueeze(1)).squeeze(1)

                with torch.no_grad():
                    qmax_stnew = Q(stnew).max(1).values
                    
                    target = rt + discount_factor * qmax_stnew * (1 - done.float())
 
                loss = criterion(target, qst)

                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                ep_loss += loss.item()
                timestep = timestep + 1

        if(timestep > 0) : ep_loss = ep_loss / timestep
        ep_losses.append(ep_loss)
        print(ep_loss)
    torch.save(Q.state_dict(), "dqn_q.pth")


In [7]:
replay_memory = MemoryReplay(1000)
network = QNetwork(8, 4)
loss = nn.functional.mse_loss
policy = EpsilonGreadyPolicy(network)
optimizer = torch.optim.Adam(network.parameters())

In [ ]:
train(env, replay_memory, network, policy, loss, optimizer)

In [19]:
torch.save(network.state_dict(), "dqn_q.pth")

In [15]:
network.load_state_dict(torch.load("dqn_q.pth"))

<All keys matched successfully>

In [ ]:
# Evaluation
policy = EpsilonGreadyPolicy(network)
policy.set_epsilon(0)

env = gym.make("LunarLander-v3", render_mode = "human")

observation, info = env.reset()

done = False

while not done:
    action = policy.take_action(observation)
    observation, reward, terminated, truncated, info = env.step(action)

    done = terminated or truncated



env.close()

In [21]:
env.close()